# Ablation Study (Multi-Trial)

4 conditions x 4 frontier models x 5 trials (temperature 0.1). Tests contribution of CPG reference and ELM simplification.

In [1]:
import pandas as pd, numpy as np, os, warnings
from scipy import stats
warnings.filterwarnings('ignore')

ABL_DIR = None
for d in ['../results/ablation_multi_trial', 'results/ablation_multi_trial']:
    if os.path.isdir(d): ABL_DIR = d; break

MODES = ['full', 'no_cpg', 'no_simplify', 'no_cpg_no_simplify']
ML = {'full': 'Simplified+CPG', 'no_cpg': 'Simplified only',
      'no_simplify': 'Raw JSON+CPG', 'no_cpg_no_simplify': 'Raw JSON only'}

def load_csv(path):
    df = pd.read_csv(path)
    for col in ['valid', 'correct', 'expected_valid']:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().str.lower() == 'true'
    return df

frames = []
for f in sorted(os.listdir(ABL_DIR)):
    if f.startswith('results-') and f.endswith('.csv'):
        frames.append(load_csv(os.path.join(ABL_DIR, f)))
data = pd.concat(frames, ignore_index=True)
models = sorted(data['model'].unique())
n_trials = data['trial'].nunique()
print(f"Loaded: {len(models)} models x {data['ablation'].nunique()} conditions x {n_trials} trials")

Loaded: 4 models x 4 conditions x 5 trials


## Mean Accuracy by Condition (5 trials, T=0.1)

In [2]:
rows = []
for model in models:
    for mode in MODES:
        mdf = data[(data['model'] == model) & (data['ablation'] == mode)]
        if len(mdf) == 0: continue
        trial_accs = mdf.groupby('trial')['correct'].mean()
        rows.append({
            'Model': model, 'Condition': ML[mode],
            'Accuracy': trial_accs.mean(), 'SD': trial_accs.std()
        })
adf = pd.DataFrame(rows)
pivot = adf.pivot(index='Condition', columns='Model', values='Accuracy')
pivot = pivot[models].reindex([ML[m] for m in MODES])
pivot.style.format('{:.1%}').background_gradient(cmap='RdYlGn', vmin=0.5, vmax=1.0).set_caption(
    'Ablation: Mean Accuracy over 5 Trials by Condition and Model')

Model,gpt-oss-120b,gpt-oss-20b,llama-3.3-70b,qwen3-32b
Condition,,,,
Simplified+CPG,89.7%,88.4%,89.0%,87.1%
Simplified only,67.7%,62.6%,57.4%,51.0%
Raw JSON+CPG,81.9%,57.4%,96.8%,83.2%
Raw JSON only,72.9%,67.7%,58.1%,57.4%


## Delta from Full Pipeline (Mean ± SD)

In [3]:
print(f"{'Model':<20s} {'Full (mean±SD)':>16s} {'No CPG':>10s} {'No Simplify':>12s} {'Neither':>10s}")
print('-' * 72)
for model in models:
    accs, sds = {}, {}
    for mode in MODES:
        sub = data[(data['model'] == model) & (data['ablation'] == mode)]
        if len(sub) > 0:
            trial_accs = sub.groupby('trial')['correct'].mean()
            accs[mode] = trial_accs.mean()
            sds[mode] = trial_accs.std()
    if 'full' not in accs: continue
    full = accs['full']
    d1 = accs.get('no_cpg', 0) - full
    d2 = accs.get('no_simplify', 0) - full
    d3 = accs.get('no_cpg_no_simplify', 0) - full
    print(f"{model:<20s} {full:>7.1%}±{sds['full']:.1%} {d1:>+9.1%} {d2:>+11.1%} {d3:>+9.1%}")

ns = data[(data['model'] == 'llama-3.3-70b') & (data['ablation'] == 'no_simplify')]
ns_sd = ns.groupby('trial')['correct'].mean().std()
print(f"\nKey findings:")
print(f"  CPG removal always hurts (Δ = -22 to -36 pp across all models)")
print(f"  Simplification benefit scales inversely with model size:")
print(f"    GPT-OSS-20B needs simplification (Δ = {accs.get('no_simplify',0)-accs.get('full',0):+.0%} without)")
llama_accs = {}
for mode in MODES:
    sub = data[(data['model'] == 'llama-3.3-70b') & (data['ablation'] == mode)]
    if len(sub) > 0: llama_accs[mode] = sub.groupby('trial')['correct'].mean().mean()
print(f"    Llama 70B bypasses it ({llama_accs.get('no_simplify',0):.1%} raw JSON, SD={ns_sd:.1%})")

Model                  Full (mean±SD)     No CPG  No Simplify    Neither
------------------------------------------------------------------------
gpt-oss-120b           89.7%±1.4%    -21.9%       -7.7%    -16.8%
gpt-oss-20b            88.4%±1.8%    -25.8%      -31.0%    -20.6%
llama-3.3-70b          89.0%±1.8%    -31.6%       +7.7%    -31.0%
qwen3-32b              87.1%±5.1%    -36.1%       -3.9%    -29.7%

Key findings:
  CPG removal always hurts (Δ = -22 to -36 pp across all models)
  Simplification benefit scales inversely with model size:
    GPT-OSS-20B needs simplification (Δ = -4% without)
    Llama 70B bypasses it (96.8% raw JSON, SD=0.0%)
